In [1]:
import sys
from pathlib import Path

# 1. Определяем корень проекта (на уровень выше текущей папки notebooks)
project_root = Path.cwd().parent

# 2. Добавляем папку src в путь поиска модулей
# Теперь Python сможет найти data_loader.py внутри src
sys.path.insert(0, str(project_root / "src"))

# 3. Импортируем функцию загрузки
from data_loader import load_data

# 4. Загружаем данные (путь к папке data тоже строим от корня)
data_path = project_root / "data"
data = load_data(data_path)

# 5. Распаковываем датасеты
train = data["train"]
test = data["test"]
submission = data["submission"]

# 6. Проверяем, что всё загрузилось
print("✅ Данные успешно загружены!")
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Submission shape: {submission.shape}")

✅ Данные успешно загружены!
Train shape: (8693, 14)
Test shape: (4277, 13)
Submission shape: (4277, 2)


In [1]:
import pandas as pd

## **Описание файлов и полей данных**
**train.csv** — персональные данные примерно двух третей (~8700) пассажиров, которые будут использоваться в качестве обучающих данных.

* **PassengerId** - Уникальный идентификатор для каждого пассажира. Каждый идентификатор имеет вид gggg_pp, где gggg обозначает группу, с которой путешествует пассажир, а pp — его номер в группе. Люди в одной группе часто являются членами одной семьи, но не всегда.
* **HomePlanet** - Планета, с которой прибыл пассажир, как правило, его планета постоянного проживания.
* **CryoSleep** - Указывает, решил ли пассажир погрузиться в анабиоз на время путешествия. Пассажиры в криосне находятся в своих каютах.
* **Cabin** - Номер каюты, в которой находится пассажир. Принимает вид deck/num/side, где side может быть P для левого борта или S для правого борта.
* **Destination** - Планета, на которую прибудет пассажир.
* **Age** - Возраст пассажира.
* **VIP** - оплатил ли пассажир специальное VIP-обслуживание во время поездки.
* **RoomService**, **FoodCourt**, **ShoppingMall**, **Spa**, **VRDeck** — сумма, которую пассажир заплатил за каждую из многочисленных роскошных услуг космического корабля «Титаник».
* **Name** - Имя и фамилия пассажира.
* **Transported** - Был ли пассажир перенесен в другое измерение. Это цель, столбец, который вы пытаетесь предсказать.

**test.csv** — персональные данные оставшейся трети (~4300) пассажиров, которые будут использоваться в качестве тестовых данных. Ваша задача — спрогнозировать значение Transported для пассажиров из этого набора.

**sample_submission.csv** — файл для отправки в правильном формате.
* **PassengerId** - Идентификатор для каждого пассажира в тестовом наборе.
* **Transported** - Цель. Для каждого пассажира предскажите, что он сделает: True или False.

План проекта:
1. Загрузка данных + быстрый EDA — баланс классов, распределения, пропуски, утечки.
2. Фиксация схемы валидации — StratifiedKFold или train/valid/holdout.
3. Feature engineering — новые признаки, циклические кодировки, агрегации.
4. Обработка пропусков и предобработка — внутри ColumnTransformer.
5. Pipeline baseline на логистической регрессии — быстрая проверка, есть ли сигнал.
6. Дерево решений + случайный лес — подбор гиперпараметров на валидации, сравнение метрик.
7. Отбор признаков — серия экспериментов: без отбора, топ‑N, SelectKBest; выбор лучшего по валидации.
8. Градиентный бустинг — LightGBM/XGBoost/CatBoost; подбор гиперпараметров.
9. Ансамбли (опционально) — Voting/Stacking, если есть время и прирост на валидации.
10. Финальное обучение — на train + valid (или всех фолдах CV), предсказание на test, сабмит.
11. Логирование — сохранение параметров, признаков, метрик для каждого эксперимента.

In [3]:
train['Transported'].value_counts(normalize = True)

Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64